# Базовый EDA: `tgbn-trade` и `tgbn-genre`

Смотрим на поток рёбер, который видит модель (`src, dst, t, msg` из `get_TemporalData`):
базовые статистики, устройство **времени** и **message**, распределение **вершин** друг между другом (степени, bipartite-структура).

Конвенции: Polars, Plotly, русский, kernel `tgb`.

In [1]:
import torch, polars as pl, numpy as np
import plotly.express as px, plotly.graph_objects as go
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset

def load_edges(name):
    ds = PyGNodePropPredDataset(name=name, root="datasets")
    d = ds.get_TemporalData()
    msg = d.msg.numpy()
    edges = pl.DataFrame({
        "src": d.src.numpy(), "dst": d.dst.numpy(),
        "t": d.t.numpy().astype("int64"),
        "msg": msg.reshape(-1) if msg.shape[-1] == 1 else msg[:, 0],
    })
    meta = dict(name=name, num_nodes=int(d.num_nodes), num_edges=edges.height,
                num_classes=int(ds.num_classes), msg_dim=int(d.msg.size(-1)),
                n_train=int(ds.train_mask.sum()), n_val=int(ds.val_mask.sum()),
                n_test=int(ds.test_mask.sum()))
    return edges, meta

D = {}
for nm in ["tgbn-trade", "tgbn-genre"]:
    e, m = load_edges(nm)
    D[nm] = {"edges": e, "meta": m}
    print(m)

{'name': 'tgbn-trade', 'num_nodes': 255, 'num_edges': 468245, 'num_classes': 255, 'msg_dim': 1, 'n_train': 337224, 'n_val': 73170, 'n_test': 57851}


{'name': 'tgbn-genre', 'num_nodes': 1505, 'num_edges': 17858395, 'num_classes': 513, 'msg_dim': 1, 'n_train': 12500878, 'n_val': 2678760, 'n_test': 2678757}


## 1. Базовые статистики

In [3]:
pl.Config.set_tbl_rows(30)  # показываем все строки, без многоточия

def base_stats(name):
    e, m = D[name]["edges"], D[name]["meta"]
    src_u, dst_u = e["src"].unique(), e["dst"].unique()
    overlap = len(set(src_u.to_list()) & set(dst_u.to_list()))
    n_pairs = e.select(["src", "dst"]).n_unique()
    return {
        "узлов": m["num_nodes"], "классов (item)": m["num_classes"],
        "рёбер": m["num_edges"], "msg_dim": m["msg_dim"],
        "train/val/test": f"{m['n_train']:,}/{m['n_val']:,}/{m['n_test']:,}",
        "уник. timestamps": e["t"].n_unique(),
        "уник. src": src_u.len(), "уник. dst": dst_u.len(),
        "src∩dst (bipartite?)": overlap,
        "уник. пар (src,dst)": n_pairs,
        "плотность пар": round(n_pairs / (src_u.len() * dst_u.len()), 4),
    }

stats = {nm: base_stats(nm) for nm in D}
tab = pl.DataFrame({"метрика": list(stats["tgbn-trade"].keys()),
                    "tgbn-trade": [str(v) for v in stats["tgbn-trade"].values()],
                    "tgbn-genre": [str(v) for v in stats["tgbn-genre"].values()]})
print(tab)

shape: (11, 3)
┌──────────────────────┬───────────────────────┬────────────────────────────────┐
│ метрика              ┆ tgbn-trade            ┆ tgbn-genre                     │
│ ---                  ┆ ---                   ┆ ---                            │
│ str                  ┆ str                   ┆ str                            │
╞══════════════════════╪═══════════════════════╪════════════════════════════════╡
│ узлов                ┆ 255                   ┆ 1505                           │
│ классов (item)       ┆ 255                   ┆ 513                            │
│ рёбер                ┆ 468245                ┆ 17858395                       │
│ msg_dim              ┆ 1                     ┆ 1                              │
│ train/val/test       ┆ 337,224/73,170/57,851 ┆ 12,500,878/2,678,760/2,678,757 │
│ уник. timestamps     ┆ 31                    ┆ 4187046                        │
│ уник. src            ┆ 254                   ┆ 992                            │
│

## 2. Время

In [4]:
from datetime import datetime, timezone
for nm in D:
    t = D[nm]["edges"]["t"]
    lo, hi = int(t.min()), int(t.max())
    print(f"{nm}: t in [{lo}, {hi}], span={hi-lo}, distinct={t.n_unique()}")
    # пробуем трактовать как unix-секунды
    try:
        print("   как даты:", datetime.fromtimestamp(lo, timezone.utc).date(),
              "→", datetime.fromtimestamp(hi, timezone.utc).date())
    except Exception as ex:
        print("   не unix-время:", ex)
print("\ntrade уникальные t:", sorted(D["tgbn-trade"]["edges"]["t"].unique().to_list()))

tgbn-trade: t in [1986, 2016], span=30, distinct=31
   как даты: 1970-01-01 → 1970-01-01
tgbn-genre: t in [1108357203, 1245461220], span=137104017, distinct=4187046
   как даты: 2005-02-14 → 2009-06-20

trade уникальные t: [1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]


In [10]:
# trade — рёбер по годам (дискретные срезы); год как категория, по одному графику на ячейку
tr = (D["tgbn-trade"]["edges"].group_by("t").len().sort("t")
        .with_columns(pl.col("t").cast(pl.Utf8).alias("год"),
                      pl.col("len").cast(pl.Int64).alias("рёбер")))
print("trade: рёбер/год — min/median/max:", tr["рёбер"].min(), tr["рёбер"].median(), tr["рёбер"].max())
fig = px.bar(tr.to_pandas(), x="год", y="рёбер", title="tgbn-trade: число рёбер по годам")
fig.update_xaxes(type="category")
fig.show()

trade: рёбер/год — min/median/max: 9330 16475.0 19371


In [11]:
# genre — рёбер по месяцам (непрерывное время)
ge = D["tgbn-genre"]["edges"].with_columns(pl.from_epoch(pl.col("t"), time_unit="s").alias("dt"))
gm = (ge.with_columns(pl.col("dt").dt.strftime("%Y-%m").alias("месяц"))
        .group_by("месяц").len().sort("месяц")
        .with_columns(pl.col("len").cast(pl.Int64).alias("рёбер")))
print("genre: рёбер/timestamp в среднем:",
      round(D["tgbn-genre"]["meta"]["num_edges"] / ge["t"].n_unique(), 2), "| месяцев:", gm.height)
fig = px.line(gm.to_pandas(), x="месяц", y="рёбер", markers=True,
              title="tgbn-genre: число рёбер по месяцам")
fig.show()

genre: рёбер/timestamp в среднем: 4.27 | месяцев: 53


## 3. Message (вес взаимодействия, `msg_dim=1`)

In [6]:
qs = [0.0, 0.5, 0.9, 0.99, 0.999, 1.0]
rows = []
for nm in D:
    msg = D[nm]["edges"]["msg"]
    r = {"датасет": nm, "min": msg.min(), "mean": round(msg.mean(), 4),
         "max": msg.max(), "<=0 доля": round((msg <= 0).mean(), 4)}
    for q in qs:
        r[f"q{q}"] = round(msg.quantile(q), 4)
    rows.append(r)
pl.Config.set_tbl_cols(20)
print(pl.DataFrame(rows))

shape: (2, 11)
┌──────────┬──────────┬────────┬─────┬─────────┬────────┬────────┬────────┬────────┬────────┬──────┐
│ датасет  ┆ min      ┆ mean   ┆ max ┆ <=0     ┆ q0.0   ┆ q0.5   ┆ q0.9   ┆ q0.99  ┆ q0.999 ┆ q1.0 │
│ ---      ┆ ---      ┆ ---    ┆ --- ┆ доля    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---  │
│ str      ┆ f64      ┆ f64    ┆ f64 ┆ ---     ┆ f64    ┆ f64    ┆ f64    ┆ f64    ┆ f64    ┆ f64  │
│          ┆          ┆        ┆     ┆ f64     ┆        ┆        ┆        ┆        ┆        ┆      │
╞══════════╪══════════╪════════╪═════╪═════════╪════════╪════════╪════════╪════════╪════════╪══════╡
│ tgbn-tra ┆ 5.6991e- ┆ 0.0144 ┆ 1.0 ┆ 0.0     ┆ 0.0    ┆ 0.0005 ┆ 0.0256 ┆ 0.2776 ┆ 0.8718 ┆ 1.0  │
│ de       ┆ 9        ┆        ┆     ┆         ┆        ┆        ┆        ┆        ┆        ┆      │
│ tgbn-gen ┆ 0.103448 ┆ 0.4405 ┆ 1.0 ┆ 0.0     ┆ 0.1034 ┆ 0.3774 ┆ 1.0    ┆ 1.0    ┆ 1.0    ┆ 1.0  │
│ re       ┆          ┆        ┆     ┆         ┆        ┆        ┆        ┆ 

In [7]:
def hist_bar(values, title, xlabel, bins=60, logx=False):
    v = np.asarray(values)
    if logx:
        v = np.log10(v); xlabel = f"log10({xlabel})"
    counts, edges = np.histogram(v, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return px.bar(x=centers, y=counts, title=title, labels={"x": xlabel, "y": "рёбер"})

tr_msg = D["tgbn-trade"]["edges"]["msg"].to_numpy()
ge_msg = D["tgbn-genre"]["edges"]["msg"].to_numpy()
hist_bar(tr_msg, "tgbn-trade: log10(msg) — нормированный вес торговли (тяжёлый хвост)", "msg", logx=True).show()
hist_bar(ge_msg, "tgbn-genre: msg — вес ([0.10, 1.0], пик у 1.0)", "msg").show()

## 4. Распределение вершин друг между другом (степени, bipartite-асимметрия)

`src` — это «отправители» (trade: нация-экспортёр; genre: **юзер**), `dst` — «получатели» (trade: нация-импортёр; genre: **жанр/item**).

In [8]:
rows = []
DEG = {}  # сохраним степени для графиков ниже
for nm in D:
    e = D[nm]["edges"]
    DEG[nm] = {}
    for side, col, other in [("src", "src", "dst"), ("dst", "dst", "src")]:
        g = e.group_by(col).agg(pl.len().alias("deg"), pl.col(other).n_unique().alias("partners"))
        DEG[nm][side] = g
        rows.append({"датасет": nm, "сторона": side, "узлов": g.height,
                     "deg медиана": int(g["deg"].median()), "deg max": int(g["deg"].max()),
                     "партнёров медиана": int(g["partners"].median()),
                     "партнёров max": int(g["partners"].max())})
print(pl.DataFrame(rows))

shape: (4, 7)
┌────────────┬─────────┬───────┬─────────────┬─────────┬───────────────────┬───────────────┐
│ датасет    ┆ сторона ┆ узлов ┆ deg медиана ┆ deg max ┆ партнёров медиана ┆ партнёров max │
│ ---        ┆ ---     ┆ ---   ┆ ---         ┆ ---     ┆ ---               ┆ ---           │
│ str        ┆ str     ┆ i64   ┆ i64         ┆ i64     ┆ i64               ┆ i64           │
╞════════════╪═════════╪═══════╪═════════════╪═════════╪═══════════════════╪═══════════════╡
│ tgbn-trade ┆ src     ┆ 254   ┆ 1267        ┆ 6116    ┆ 141               ┆ 242           │
│ tgbn-trade ┆ dst     ┆ 254   ┆ 1694        ┆ 5673    ┆ 147               ┆ 234           │
│ tgbn-genre ┆ src     ┆ 992   ┆ 9766        ┆ 338602  ┆ 130               ┆ 397           │
│ tgbn-genre ┆ dst     ┆ 513   ┆ 4252        ┆ 2111205 ┆ 181               ┆ 975           │
└────────────┴─────────┴───────┴─────────────┴─────────┴───────────────────┴───────────────┘


In [9]:
def rank_deg(nm, side, label):
    d = DEG[nm][side]["deg"].sort(descending=True).to_numpy()
    return pl.DataFrame({"rank": np.arange(1, len(d) + 1), "deg": d, "сторона": [label] * len(d)})

# genre: user (src) vs item/жанр (dst)
ge_rd = pl.concat([rank_deg("tgbn-genre", "src", "user (src)"),
                   rank_deg("tgbn-genre", "dst", "жанр/item (dst)")])
px.line(ge_rd.to_pandas(), x="rank", y="deg", color="сторона", log_x=True, log_y=True,
        title="tgbn-genre: rank–degree (log-log) — item'ы тяжелее юзеров",
        labels={"deg": "число рёбер (degree)"}).show()

# trade: src vs dst (симметрично)
tr_rd = pl.concat([rank_deg("tgbn-trade", "src", "нация-экспортёр (src)"),
                   rank_deg("tgbn-trade", "dst", "нация-импортёр (dst)")])
px.line(tr_rd.to_pandas(), x="rank", y="deg", color="сторона", log_x=True, log_y=True,
        title="tgbn-trade: rank–degree (log-log) — стороны симметричны",
        labels={"deg": "число рёбер (degree)"}).show()

## Выводы EDA

**Структура.** `tgbn-trade` — почти симметричный граф наций (src∩dst = 253/254, плотность пар 0.53). `tgbn-genre` — чистый bipartite: **992 юзера → 513 жанров** (src∩dst = 0), плотность пар 0.26 (юзер в среднем касается ~130 из 513 жанров).

**Время.** Радикально разное: trade — **31 годовой дискретный срез** (1986–2016), ~9–19k рёбер/год. genre — **почти непрерывное** unix-время (2005-02 → 2009-06, 4.19M уник. timestamps, ~4.3 ребра/timestamp), активность по месяцам неравномерна.

**Message** (`dim=1`, нормированный вес ∈ [0,1]). trade — **крайне тяжёлый хвост** (медиана 5e-4, mean 0.014, max 1.0) → нужна log-шкала. genre — ограничен снизу ~0.10, медиана 0.38, заметная масса у 1.0.

**Распределение вершин.** trade — плотно и почти равномерно (медиана партнёров 141/254). genre — **сильная асимметрия**: item'ы малочисленны, но «тяжёлые» — топ-жанр в 2.11M рёбер и затронут 975/992 юзерами (глобальный prior, ср. P1: top-10 = 42% массы); юзеры медианно 9766 рёбер по ~130 жанрам. Степени item-стороны куда более skewed, чем user-стороны.

**Для исследования.** Подтверждает рычаг асимметрии: item'ы всегда «тёплые» (высокая степень) ⇒ их представления насыщены и холодные юзеры могут заимствовать item-side структуру. Плотность пар 26% и малое N_item делают плотную `N_user × N_item` матрицу дешёвой (scalar: genre ~2 MB) и осмысленной.